In [ ]:
# ---------------------------------------------
# Created By: Carlos Galvez
# Date: 22/03/2025
# ---------------------------------------------
import sys
import math
from OpenGL.GL import *
from OpenGL.GLUT import *
from OpenGL.GLU import *

# ---------------------------------------------
# Parámetros globales
# ---------------------------------------------
data_dict = {}       # Diccionario para almacenar la cantidad de estaciones por marca
dcp_dict = {}        # Diccionario para almacenar la cantidad de estaciones por código postal
colors_dict = {}     # Diccionario para asociar un color a cada marca
percentages = {}     # Diccionario para almacenar los porcentajes ajustados por marca
window_width = 1400  # Anchura de la ventana de visualización
window_height = 800  # Altura de la ventana de visualización

# ---------------------------------------------
# Leer datos desde el archivo CSV
# ---------------------------------------------
def leer_datos(filename):
    global data_dict, dcp_dict
    try:
        with open(filename, 'r', encoding='utf-8') as file:
            for line_num, line in enumerate(file, start=1):
                print(f"[Línea {line_num}] {line.strip()}")
                partes = line.strip().split(';')
                if len(partes) != 4:
                    print(f"❗ Línea mal formateada en la línea {line_num}")
                    continue
                marca = partes[3].strip().upper()
                cp = partes[1].strip()
                data_dict[marca] = data_dict.get(marca, 0) + 1
                dcp_dict[cp] = dcp_dict.get(cp, 0) + 1
    except Exception as e:
        print(f"❌ Error al leer datos: {e}")
        sys.exit(1)

# ---------------------------------------------
# Calcular porcentajes enteros que sumen 100
# ---------------------------------------------
def calcular_porcentajes():
    global percentages
    total = sum(data_dict.values())
    marcas = list(data_dict.keys())
    reales = [(marca, (data_dict[marca] / total) * 100) for marca in marcas]
    enteros = [(marca, int(p)) for marca, p in reales]
    residuales = [(marca, p - int(p)) for marca, p in reales]

    suma_actual = sum(e for _, e in enteros)
    diferencia = 100 - suma_actual

    # Ordenar por mayor residuo para distribuir el sobrante
    residuales.sort(key=lambda x: x[1], reverse=True)
    for i in range(diferencia):
        marca = residuales[i][0]
        for j in range(len(enteros)):
            if enteros[j][0] == marca:
                enteros[j] = (marca, enteros[j][1] + 1)
                break

    percentages = dict(enteros)

# ---------------------------------------------
# Dibujar texto
# ---------------------------------------------
def draw_text(text, x, y, font):
    glRasterPos2f(x, y)
    for ch in text:
        glutBitmapCharacter(font, ord(ch))

# ---------------------------------------------
# Dibujar gráfico circular con leyenda
# ---------------------------------------------
def draw_pie_chart():
    total = sum(data_dict.values())
    angle_start = 0.0
    radius = 0.45
    cx, cy = -0.4, -0.2 #Línea donde se define la posición del centro del gráfico

    marcas = list(data_dict.keys())
    for i, marca in enumerate(marcas):
        colors_dict[marca] = (math.sin(i) % 1, math.cos(i) % 1, (i * 0.3) % 1)

    for marca in marcas:
        porcentaje_entero = percentages.get(marca, 0)
        angle_end = angle_start + (porcentaje_entero / 100.0) * 360.0
        mid_angle = (angle_start + angle_end) / 2

        glBegin(GL_TRIANGLE_FAN)
        glColor3fv(colors_dict[marca])
        glVertex2f(cx, cy)
        for angle in range(int(angle_start), int(angle_end)+1):
            rad = math.radians(angle)
            x = cx + math.cos(rad) * radius
            y = cy + math.sin(rad) * radius
            glVertex2f(x, y)
        glEnd()

        # Texto de porcentaje
        percent_text = f"{porcentaje_entero}%"
        rad = math.radians(mid_angle)
        tx = cx + math.cos(rad) * (radius * 0.6)
        ty = cy + math.sin(rad) * (radius * 0.6)
        glColor3f(1, 1, 1)
        draw_text(percent_text, tx, ty, GLUT_BITMAP_HELVETICA_12)

        angle_start = angle_end

    draw_legend()

# ---------------------------------------------
# Dibujar leyenda a la izquierda del círculo
# ---------------------------------------------
def draw_legend():
    x_legend = -0.95
    y_legend = 0.85
    box_size = 0.04
    spacing = 0.07

    glColor3f(0, 0, 0)
    draw_text("Distribución de Gasolineras por Marca", x_legend, y_legend + 0.1, GLUT_BITMAP_HELVETICA_18)

    for i, (marca, color) in enumerate(colors_dict.items()):
        y = y_legend - i * spacing
        glColor3fv(color)
        glBegin(GL_POLYGON)
        glVertex2f(x_legend, y)
        glVertex2f(x_legend + box_size, y)
        glVertex2f(x_legend + box_size, y - box_size)
        glVertex2f(x_legend, y - box_size)
        glEnd()

        glColor3f(0, 0, 0)
        label = f"{marca} ({data_dict[marca]}) = {percentages.get(marca, 0)}%"
        draw_text(label, x_legend + box_size + 0.02, y - 0.025, GLUT_BITMAP_HELVETICA_12)

# ---------------------------------------------
# Dibujar gráfico de barras con top 15 CP
# ---------------------------------------------
def draw_bar_chart():
    sorted_cps = sorted(dcp_dict.items(), key=lambda x: int(x[0]))
    top_15 = sorted_cps[:15] if len(sorted_cps) > 15 else sorted_cps
    cps = [int(k) for k, _ in top_15]
    vals = [v for _, v in top_15]

    min_val, max_val = 0, max(vals)
    bar_width = 0.04
    spacing = 0.06

    base_x = 0.1 #Coordenada horizontal inicial del gráfico de barras
    base_y = -0.8 #Coordenada vertical (más alto o más bajo)

    for i, (cp, val) in enumerate(zip(cps, vals)):
        x = base_x + i * spacing
        height = (val / max_val) * 0.6

        r = (math.sin(i) * 0.5 + 0.5) % 1
        g = (math.cos(i) * 0.5 + 0.5) % 1
        b = ((i * 0.3) % 1)
        glColor3f(r, g, b)

        glBegin(GL_POLYGON)
        glVertex2f(x, base_y)
        glVertex2f(x + bar_width, base_y)
        glVertex2f(x + bar_width, base_y + height)
        glVertex2f(x, base_y + height)
        glEnd()

        glColor3f(0, 0, 0)
        draw_text(str(cp), x, base_y - 0.05, GLUT_BITMAP_HELVETICA_10)

    glColor3f(0.8, 0.8, 0.8)
    for i in range(1, 6):
        y = base_y + i * 0.12
        glBegin(GL_LINES)
        glVertex2f(base_x - 0.02, y)
        glVertex2f(base_x + len(cps) * spacing, y)
        glEnd()

    glColor3f(0, 0, 0)
    draw_text("Top 15 Códigos Postales con Más Estaciones", 0.3, -0.15, GLUT_BITMAP_HELVETICA_18)

# ---------------------------------------------
# Renderizado
# ---------------------------------------------
def display():
    glClear(GL_COLOR_BUFFER_BIT)
    glLoadIdentity()
    draw_pie_chart()
    draw_bar_chart()
    draw_text("Pulsa ESC para salir", 0.6, -0.95, GLUT_BITMAP_HELVETICA_12)
    glutSwapBuffers()

# ---------------------------------------------
# Inicialización
# ---------------------------------------------
def init():
    glClearColor(1.0, 1.0, 1.0, 1.0)
    glMatrixMode(GL_PROJECTION)
    glLoadIdentity()
    gluOrtho2D(-1, 1, -1, 1)
# ---------------------------------------------
# Acción Cerrar Ventana al pulsar tecla ESC
# ---------------------------------------------
def keyboard(key, x, y):
    if key == b'\x1b':  # ESC
        try:
            glutLeaveMainLoop()  # salir del bucle principal
        except:
            sys.exit()
# ---------------------------------------------
# Main
# ---------------------------------------------
def main():
    leer_datos("gas_stations.csv")
    calcular_porcentajes()
    glutInit(sys.argv)
    glutInitDisplayMode(GLUT_RGBA | GLUT_DOUBLE)
    glutInitWindowSize(window_width, window_height)
    glutCreateWindow(b"Graficos OpenGL - Circular y Barras")
    init()
    glutDisplayFunc(display)
    glutKeyboardFunc(keyboard)
    glutMainLoop()

if __name__ == "__main__":
    main()


[Línea 1] ﻿MADRID;28805;C.C. LA DEHESA, AUTOVIA MADRID-BARCELONA km 34,0;ALCAMPO
[Línea 2] MADRID;28770;CALLE C.C. ALCAMPO, C/. TEJERA, 2;ALCAMPO
[Línea 3] MADRID;28000;CALLE JOSE PAULETE, ;ALCAMPO
[Línea 4] MADRID;28922;AVENIDA EUROPA, S/N;ALCAMPO
[Línea 5] MADRID;28108;AVENIDA OLIMPICA, 9;ALCAMPO
[Línea 6] MADRID;28802;CALLE AVENIDA MADRID ESQUINA CARLOS III, SN;GALP
[Línea 7] MADRID;28320;CARRETERA M-506 KM. 24;GALP
[Línea 8] MADRID;28691;AVENIDA ESPAÑA, 1;ALCAMPO
[Línea 9] MADRID;28609;CARRETERA M-600 KM. 44;GALP
[Línea 10] MADRID;28609;CARRETERA M-600 KM. 44;GALP
[Línea 11] MADRID;28980;CALLE BUENAVISTA, 2;GALP
[Línea 12] MADRID;28914;CALLE SEVERO OCHOA ESQUINA CALLE JULIO PALACIOS, S/N;SHELL
[Línea 13] MADRID;28400;CALLE MATADERO, CARREFOUR LOS VALLES, S/N;GALP
[Línea 14] MADRID;28912;AVENIDA FUENLABRADA, 16;BP
[Línea 15] MADRID;28100;CARRETERA ANTIGUA NACIONAL I KM. 15,5;GALP
[Línea 16] MADRID;28850;CARRETERA LOECHES-AJALVIR, KM. 12,500;ALCAMPO
[Línea 17] MADRID;28051;CALLE ADOL